<a href="https://colab.research.google.com/github/HazardMG-py/xgCCA-SSG/blob/realfeature/xgcca_ssg_colab_full.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# xgCCA-SSG (Colab) — Full Pipeline, Pinned Deps (Torch 2.3.1 + cu121)
This notebook installs a compatible stack and runs the full xgCCA-SSG pipeline:
1) Install dependencies (Torch 2.3.1 + cu121, DGL torch-2.3/cu121, PyG 2.6.1, OGB, torchdata 0.7.1)
2) Create project tree and write source files
3) Train self-supervised model + linear probe
4) Optional one-vs-rest AUROC
5) Parameter sweeps and plots


In [ ]:
# 📦 Install dependencies (pinned, Torch 2.3.1 + cu121 + matching DGL)
!pip -q uninstall -y dgl torch torchvision torchaudio torch_geometric torchdata torchtune \
  pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv ogb

# 1) Torch 2.3.1 (CUDA 12.1 wheels)
!pip -q install torch==2.3.1+cu121 torchvision==0.18.1+cu121 torchaudio==2.3.1+cu121 \
  --index-url https://download.pytorch.org/whl/cu121

# 2) DGL built for Torch 2.3 + cu121 (GraphBolt-compatible)
!pip -q install dgl -f https://data.dgl.ai/wheels/torch-2.3/cu121/repo.html

# 3) PyG pinned to Torch 2.3.1+cu121
!pip -q install torch-geometric==2.6.1 \
  --extra-index-url https://data.pyg.org/whl/torch-2.3.1+cu121.html

# 4) OGB + torchdata (no-deps so we don't touch torch)
!pip -q install ogb
!pip -q install torchdata==0.7.1 --no-deps

# 🔁 Force runtime restart so the new Torch is actually used
import os; os.kill(os.getpid(), 9)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.9/780.9 MB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 131.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 116.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 98.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 61.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 110.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 18.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
# ✅ Sanity check versions
import torch, pkgutil
print('Torch:', torch.__version__)
import dgl; print('DGL:', dgl.__version__)
import torchdata; print('torchdata:', torchdata.__version__)
print('PyG present?:', pkgutil.find_loader('torch_geometric') is not None)
print('OGB present?:', pkgutil.find_loader('ogb') is not None)
print('torchtune present?:', pkgutil.find_loader('torchtune') is not None)


Torch: 2.3.1+cu121
Setting the default backend to "pytorch". You can change it in the ~/.dgl/config.json file or export the DGLBACKEND environment variable.  Valid options are: pytorch, mxnet, tensorflow (all lowercase)


DGL backend not selected or invalid.  Assuming PyTorch for now.


DGL: 2.5.0+cu121
torchdata: 0.7.1
PyG present?: True
OGB present?: True
torchtune present?: False


/tmp/ipython-input-2101395016.py:6: DeprecationWarning: 'pkgutil.find_loader' is deprecated and slated for removal in Python 3.14; use importlib.util.find_spec() instead
  print('PyG present?:', pkgutil.find_loader('torch_geometric') is not None)
/tmp/ipython-input-2101395016.py:7: DeprecationWarning: 'pkgutil.find_loader' is deprecated and slated for removal in Python 3.14; use importlib.util.find_spec() instead
  print('OGB present?:', pkgutil.find_loader('ogb') is not None)
/tmp/ipython-input-2101395016.py:8: DeprecationWarning: 'pkgutil.find_loader' is deprecated and slated for removal in Python 3.14; use importlib.util.find_spec() instead
  print('torchtune present?:', pkgutil.find_loader('torchtune') is not None)


In [2]:
# 🧮 GPU details
import torch, platform
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
print('Platform:', platform.platform())


CUDA available: True
GPU name: Tesla T4
Platform: Linux-6.6.105+-x86_64-with-glibc2.35


In [3]:
# 🗂️ Create project tree
import os
ROOT = "/content/xgcca_ssg"
os.makedirs(ROOT + "/src", exist_ok=True)
os.makedirs(ROOT + "/plots", exist_ok=True)
print('Project root:', ROOT)


Project root: /content/xgcca_ssg


## Write source files

In [4]:
%%writefile /content/xgcca_ssg/src/aug.py
import torch
import dgl

def random_aug(graph, x, feat_drop_rate=0.2, edge_drop_rate=0.2):
    """
    Random edge drop (preserves edge data) + per-dimension feature masking.
    Returns: (aug_graph, x_aug)
    """
    device = x.device
    E = graph.num_edges()

    if E > 0:
        keep = (torch.rand(E, device=device) > edge_drop_rate)
        if keep.sum() == 0:
            keep = torch.ones(E, dtype=torch.bool, device=device)
        src, dst = graph.edges()
        src, dst = src[keep], dst[keep]
        aug = dgl.graph((src, dst), num_nodes=graph.num_nodes(), device=device)
        for k, v in graph.edata.items():
            aug.edata[k] = v[keep].to(device)
    else:
        aug = dgl.graph(([], []), num_nodes=graph.num_nodes(), device=device)

    x_aug = x.clone()
    D = x.shape[1]
    feat_keep = (torch.rand(D, device=device) > feat_drop_rate)
    if feat_keep.sum() == 0:
        feat_keep[torch.randint(0, D, (1,), device=device)] = True
    x_aug[:, ~feat_keep] = 0
    return aug, x_aug


Writing /content/xgcca_ssg/src/aug.py


In [5]:
%%writefile /content/xgcca_ssg/src/config.py
import argparse
import torch
from IPython import get_ipython

def get_args():
    p = argparse.ArgumentParser(description='xgCCA-SSG')
    p.add_argument('--dataname', type=str, default='cora',
                   choices=['cora','citeseer','pubmed','photo','comp','cs','physics','arxiv','reddit'])
    p.add_argument('--gpu', type=int, default=0)
    p.add_argument('--epochs', type=int, default=200)
    p.add_argument('--lr1', type=float, default=1e-3)
    p.add_argument('--lr2', type=float, default=1e-2)
    p.add_argument('--wd1', type=float, default=0.0)
    p.add_argument('--wd2', type=float, default=1e-4)
    p.add_argument('--lambd', type=float, default=1e-4)             # decorrelation loss weight
    p.add_argument('--sparsity_lambda', type=float, default=1e-4)   # sparsity penalty
    p.add_argument('--tau', type=float, default=1.0)                # Gumbel temperature
    p.add_argument('--n_layers', type=int, default=2)
    p.add_argument('--num_heads', type=int, default=4)
    p.add_argument('--der', type=float, default=0.2)                # edge drop
    p.add_argument('--dfr', type=float, default=0.2)                # feature drop
    p.add_argument('--hid_dim', type=int, default=128)
    p.add_argument('--out_dim', type=int, default=256)
    p.add_argument('--run_name', type=str, default='')
    p.add_argument('--seed', type=int, default=42)
    p.add_argument('--notes', type=str, default='')
    # AUROC / binary adapter
    p.add_argument('--binary_one_vs_rest', action='store_true')
    p.add_argument('--binary_pos_class', type=int, default=0)
    args = p.parse_args(args=[]) if 'google.colab' in str(get_ipython()) else p.parse_args()
    args.device = f'cuda:{args.gpu}' if (args.gpu != -1 and torch.cuda.is_available()) else 'cpu'
    return args


Writing /content/xgcca_ssg/src/config.py


In [6]:
%%writefile /content/xgcca_ssg/src/dataset.py
import torch, numpy as np, os, dgl
from dgl.data import CoraGraphDataset, CiteseerGraphDataset, PubmedGraphDataset
from dgl.data import AmazonCoBuyPhotoDataset, AmazonCoBuyComputerDataset
from dgl.data import CoauthorCSDataset, CoauthorPhysicsDataset
from ogb.nodeproppred import DglNodePropPredDataset
from torch_geometric.datasets import Reddit

def load_data(name='cora', path='/content/data'):
    dm = {
        'cora': CoraGraphDataset,
        'citeseer': CiteseerGraphDataset,
        'pubmed': PubmedGraphDataset,
        'photo': AmazonCoBuyPhotoDataset,
        'comp': AmazonCoBuyComputerDataset,
        'cs': CoauthorCSDataset,
        'physics': CoauthorPhysicsDataset,
        'arxiv': lambda: DglNodePropPredDataset(name='ogbn-arxiv'),
        'reddit': lambda: Reddit(root=os.path.join(path, 'Reddit')),
    }
    if name not in dm:
        raise ValueError(f"Unknown dataset: {name}")
    dataset = dm[name]()

    if name == 'reddit':
        pyg = dataset[0]
        g = dgl.graph((pyg.edge_index[0], pyg.edge_index[1]), num_nodes=pyg.num_nodes)
        feat, labels = pyg.x, pyg.y
        num_classes = dataset.num_classes
        N = g.num_nodes()
        idx = np.random.permutation(N)
        tr, va = int(N*0.1), int(N*0.1)
        train_idx = torch.tensor(idx[:tr]); val_idx = torch.tensor(idx[tr:tr+va]); test_idx = torch.tensor(idx[tr+va:])
    elif name == 'arxiv':
        g, labels = dataset[0]
        labels = labels.squeeze()
        feat = g.ndata['feat']; num_classes = dataset.num_classes
        split = dataset.get_idx_split()
        train_idx, val_idx, test_idx = split['train'], split['valid'], split['test']
    else:
        g = dataset[0]
        labels = g.ndata.pop('label', None)
        feat = g.ndata.pop('feat', None)
        num_classes = dataset.num_classes
        if name in ['cora','citeseer','pubmed']:
            train_idx = torch.nonzero(g.ndata.pop('train_mask'), as_tuple=False).squeeze()
            val_idx   = torch.nonzero(g.ndata.pop('val_mask'  ), as_tuple=False).squeeze()
            test_idx  = torch.nonzero(g.ndata.pop('test_mask' ), as_tuple=False).squeeze()
        else:
            N = g.num_nodes()
            idx = np.random.permutation(N)
            tr, va = int(N*0.1), int(N*0.1)
            train_idx = torch.tensor(idx[:tr]); val_idx = torch.tensor(idx[tr:tr+va]); test_idx = torch.tensor(idx[tr+va:])

    if feat is None or labels is None:
        raise KeyError("Features or labels missing.")
    g.ndata['feat'] = feat; g.ndata['label'] = labels
    return (g, feat, labels, num_classes, train_idx, val_idx, test_idx)


Writing /content/xgcca_ssg/src/dataset.py


In [12]:
%%writefile /content/xgcca_ssg/src/xgcca_model.py
import torch, torch.nn as nn, torch.nn.functional as F, dgl
from dgl.nn import GraphConv
from dgl.nn.functional import edge_softmax

class GumbelSubspaceSelector(nn.Module):
    def __init__(self, dim, tau=1.0):
        super().__init__(); self.logits = nn.Parameter(torch.zeros(dim)); self.tau = tau
    def forward(self, hard=False):
        eps = 1e-9; u = torch.rand_like(self.logits).clamp_(min=eps, max=1-eps); g = -torch.log(-torch.log(u))
        soft = torch.sigmoid((self.logits + g) / max(self.tau, eps))
        if hard:
            hard_mask = (soft > 0.5).float(); return hard_mask + soft - soft.detach()
        return soft

class BicliqueAttentionLayer(nn.Module):
    def __init__(self, in_dim, out_dim, num_heads=4):
        super().__init__(); assert out_dim % num_heads == 0
        self.num_heads=num_heads; self.dk=out_dim//num_heads
        self.q=nn.Linear(in_dim,out_dim,bias=False); self.k=nn.Linear(in_dim,out_dim,bias=False); self.v=nn.Linear(in_dim,out_dim,bias=False)
    def forward(self, g, x, mask=None):
        if mask is None: mask = torch.ones(x.size(1), device=x.device)
        x = x * mask.unsqueeze(0)
        Q=self.q(x).view(-1,self.num_heads,self.dk); K=self.k(x).view(-1,self.num_heads,self.dk); V=self.v(x).view(-1,self.num_heads,self.dk)
        with g.local_scope():
            g.ndata['Q']=Q; g.ndata['K']=K; g.ndata['V']=V
            g.apply_edges(lambda e:{'score':(e.src['Q']*e.dst['K']).sum(-1)/(self.dk**0.5)})
            score=g.edata['score']; H=score.size(1); alphas=[]
            for h in range(H): g.edata['s']=score[:,h]; alphas.append(edge_softmax(g, g.edata['s']))
            alpha=torch.stack(alphas,dim=1).unsqueeze(-1); g.edata['alpha']=alpha
            g.update_all(dgl.function.u_mul_e('V','alpha','m'), dgl.function.sum('m','out'))
            out=g.ndata['out'].reshape(-1,H*self.dk); return F.relu(out)

class BicliqueGCN(nn.Module):
    def __init__(self, in_dim, hid_dim, out_dim, n_layers, num_heads=4):
        super().__init__(); self.layers=nn.ModuleList()
        if n_layers==1: self.layers.append(GraphConv(in_dim,out_dim,norm='both',  allow_zero_in_degree=True))
        else:
            self.layers.append(GraphConv(in_dim,hid_dim,norm='both',  allow_zero_in_degree=True))
            for _ in range(n_layers-2): self.layers.append(BicliqueAttentionLayer(hid_dim,hid_dim,num_heads))
            self.layers.append(GraphConv(hid_dim,out_dim,norm='both',  allow_zero_in_degree=True))
    def forward(self,g,x,mask=None):
        h=x
        for layer in self.layers:
            if isinstance(layer,BicliqueAttentionLayer): h=layer(g,h,mask)
            else: h=F.relu(layer(g,h))
        return h

class xgCCA_SSG(nn.Module):
    def __init__(self,in_dim,hid_dim,out_dim,n_layers,tau=1.0,sparsity_lambda=1e-4,num_heads=4):
        super().__init__(); self.encoder=BicliqueGCN(in_dim,hid_dim,out_dim,n_layers,num_heads)
        self.proj=nn.Linear(out_dim,out_dim,bias=False); self.selector=GumbelSubspaceSelector(out_dim,tau)
        self.sparsity_lambda=sparsity_lambda
    def _norm(self,z): return (z - z.mean(0)) / (z.std(0) + 1e-6)
    def forward(self,g1,x1,g2,x2):
        h1=self.encoder(g1,x1,mask=None); h2=self.encoder(g2,x2,mask=None)
        h1=self.proj(h1); h2=self.proj(h2)
        m=self.selector(hard=False); h1=h1*m.unsqueeze(0); h2=h2*m.unsqueeze(0)
        z1=self._norm(h1); z2=self._norm(h2); return z1,z2,m.mean()
    def get_embedding(self,g,x,hard_mask=True):
        h=self.encoder(g,x,mask=None); h=self.proj(h); m=self.selector(hard=hard_mask); return h*m.unsqueeze(0)

def cca_loss(z1,z2,lambd=1e-4):
    N=z1.size(0); c12=(z1.T@z2)/N; c11=(z1.T@z1)/N; c22=(z2.T@z2)/N
    I=torch.eye(c12.size(0),device=z1.device); loss_inv=-torch.diagonal(c12).sum()
    loss_dec=(c11-I).pow(2).sum()+(c22-I).pow(2).sum(); return loss_inv + lambd*loss_dec


Overwriting /content/xgcca_ssg/src/xgcca_model.py


In [8]:
%%writefile /content/xgcca_ssg/src/utils.py
import os, csv, torch, numpy as np, random
def setup_seed(seed):
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    np.random.seed(seed); random.seed(seed)
    torch.backends.cudnn.deterministic = True
def save_checkpoint(model, log_dir, epoch):
    path = os.path.join(log_dir, f'model_epoch_{epoch}.pth')
    torch.save(model.state_dict(), path)
    print(f"Saved checkpoint → {path}")
def log_results(log_dir, msg):
    with open(os.path.join(log_dir, 'results.txt'), 'a') as f: f.write(msg + '\n')
def csv_path(log_dir): return os.path.join(log_dir, "sweep_results.csv")
def init_csv(log_dir, fieldnames):
    p = csv_path(log_dir); new = not os.path.exists(p)
    with open(p, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        if new: w.writeheader()
def append_csv(log_dir, row: dict, fieldnames=None):
    p = csv_path(log_dir)
    if fieldnames is None: fieldnames = list(row.keys())
    with open(p, "a", newline="") as f:
        csv.DictWriter(f, fieldnames=fieldnames).writerow(row)
class EpochTimer:
    from time import time
    def __enter__(self): from time import time; self.t0 = time(); return self
    def __exit__(self, *exc): from time import time; self.dt = time() - self.t0


Writing /content/xgcca_ssg/src/utils.py


In [15]:
%%writefile /content/xgcca_ssg/src/main.py
import os, torch, torch.nn as nn, torch.nn.functional as F, dgl, numpy as np
from datetime import datetime
from src.config import get_args
from src.dataset import load_data
from src.xgcca_model import xgCCA_SSG, cca_loss
from src.aug import random_aug
from src.utils import setup_seed, save_checkpoint, log_results, init_csv, append_csv, csv_path, EpochTimer

class LinearProbe(nn.Module):
    def __init__(self, in_dim, num_classes): super().__init__(); self.lin = nn.Linear(in_dim, num_classes)
    def forward(self, x): return self.lin(x)
@torch.no_grad()
def accuracy(logits, labels): return (logits.argmax(1) == labels).float().mean().item()
def _safe_auroc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    if set(np.unique(y_true)) != {0,1}: return None
    pos = (y_true == 1); neg = (y_true == 0)
    n_pos, n_neg = pos.sum(), neg.sum()
    if n_pos == 0 or n_neg == 0: return None
    order = y_score.argsort(); ranks = order.argsort()
    sum_ranks_pos = ranks[pos].sum(); U = sum_ranks_pos - n_pos*(n_pos-1)/2.0
    return float(U/(n_pos*n_neg))
def eval_linear_probe(embeds, labels, train_idx, val_idx, test_idx, args):
    device = embeds.device; C = int(labels.max().item()+1)
    clf = LinearProbe(embeds.size(1), C).to(device)
    opt = torch.optim.Adam(clf.parameters(), lr=args.lr2, weight_decay=args.wd2)
    xtr, ytr = embeds[train_idx], labels[train_idx]
    xva, yva = embeds[val_idx], labels[val_idx]
    xte, yte = embeds[test_idx], labels[test_idx]
    best_val, best_test, best_auroc = 0.0, 0.0, None
    for _ in range(300):
        clf.train(); opt.zero_grad(); loss = F.cross_entropy(clf(xtr), ytr); loss.backward(); opt.step()
        clf.eval()
        with torch.no_grad():
            va = accuracy(clf(xva), yva)
            te_logits = clf(xte)
            te = accuracy(te_logits, yte)
            if C==2:
                prob1 = F.softmax(te_logits, 1)[:,1].cpu().numpy(); au = _safe_auroc(yte.cpu().numpy(), prob1)
            else:
                au = None
        if va >= best_val: best_val, best_test, best_auroc = va, te, au
    return best_val, best_test, best_auroc
def train_ssl(args, model, graph, feat, log_dir):
    device = feat.device; opt = torch.optim.Adam(model.parameters(), lr=args.lr1, weight_decay=args.wd1)
    graph = dgl.add_self_loop(dgl.remove_self_loop(graph))
    epoch_times, last_sel, last_loss = [], None, None
    for epoch in range(1, args.epochs+1):
        with EpochTimer() as t:
            model.train(); opt.zero_grad()
            g1,x1 = random_aug(graph, feat, args.dfr, args.der)
            g2,x2 = random_aug(graph, feat, args.dfr, args.der)
            g1 = dgl.add_self_loop(dgl.remove_self_loop(g1))
            g2 = dgl.add_self_loop(dgl.remove_self_loop(g2))
            z1,z2,s = model(g1,x1,g2,x2)
            loss = cca_loss(z1,z2,args.lambd) + args.sparsity_lambda*s
            loss.backward(); opt.step()
        epoch_times.append(t.dt); last_loss=float(loss.item())
        with torch.no_grad(): last_sel=int((model.selector(hard=True)>0.5).sum().item())
        msg=f"Epoch {epoch:03d} | loss {last_loss:.4f} | |S| {last_sel} | {t.dt:.3f}s"; print(msg); log_results(log_dir, msg)
        if epoch%50==0: save_checkpoint(model, log_dir, epoch)
    return model, sum(epoch_times)/len(epoch_times), sum(epoch_times), last_sel, last_loss
def main():
    args = get_args(); setup_seed(args.seed)
    device = torch.device(args.device)
    log_dir = f"/content/xgcca_ssg/logs/{args.dataname}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"; os.makedirs(log_dir, exist_ok=True)
    log_results(log_dir, str(args))
    fields = ["timestamp","run_name","dataname","seed","epochs","hid_dim","out_dim","n_layers","num_heads",
              "der","dfr","lambd","sparsity_lambda","tau","lr1","wd1","lr2","wd2",
              "avg_sec_per_epoch","total_train_sec","selected_dims","val_acc","test_acc","auroc",
              "binary_one_vs_rest","binary_pos_class","notes"]
    init_csv(log_dir, fields)
    g, feat, labels, num_classes, tr, va, te = load_data(args.dataname)
    g, feat = g.to(device), feat.to(device)
    if (labels is not None) and args.binary_one_vs_rest:
        C = int(labels.max().item()+1); pos = args.binary_pos_class
        if pos<0 or pos>=C: raise ValueError(f"--binary_pos_class in [0,{C-1}]")
        labels = (labels==pos).long().to(device)
        p_tr = float((labels[tr]==1).float().mean().item())
        m = f"[OvR] pos_class={pos} | train pos rate={p_tr:.3f}"; print(m); log_results(log_dir, m)
    model = xgCCA_SSG(feat.size(1), args.hid_dim, args.out_dim, args.n_layers,
                      tau=args.tau, sparsity_lambda=args.sparsity_lambda, num_heads=args.num_heads).to(device)
    model, avg_ep, total_sec, sel_dims, last_loss = train_ssl(args, model, g, feat, log_dir)
    if labels is None: raise ValueError("Dataset needs labels for linear probe.")
    with torch.no_grad():
      embeds = model.get_embedding(dgl.add_self_loop(dgl.remove_self_loop(g)), feat, hard_mask=True)
    embeds = embeds.detach()
    val_acc, test_acc, test_auroc = eval_linear_probe(embeds, labels.to(device), tr.to(device), va.to(device), te.to(device), args)
    print(f"[LinearProbe] Val {val_acc*100:.2f}% | Test {test_acc*100:.2f}% | AUROC {'' if test_auroc is None else round(test_auroc*100,2)}")
    log_results(log_dir, f"[LP] val={val_acc:.4f} test={test_acc:.4f} auroc={test_auroc}")
    row = {"timestamp": datetime.now().isoformat(timespec="seconds"),"run_name": args.run_name,"dataname": args.dataname,
           "seed": args.seed,"epochs": args.epochs,"hid_dim": args.hid_dim,"out_dim": args.out_dim,"n_layers": args.n_layers,
           "num_heads": args.num_heads,"der": args.der,"dfr": args.dfr,"lambd": args.lambd,"sparsity_lambda": args.sparsity_lambda,
           "tau": args.tau,"lr1": args.lr1, "wd1": args.wd1, "lr2": args.lr2, "wd2": args.wd2,
           "avg_sec_per_epoch": round(avg_ep,4),"total_train_sec": round(total_sec,2),"selected_dims": sel_dims,
           "val_acc": round(val_acc*100,2),"test_acc": round(test_acc*100,2),"auroc": "" if test_auroc is None else round(test_auroc*100,2),
           "binary_one_vs_rest": int(args.binary_one_vs_rest),"binary_pos_class": args.binary_pos_class,"notes": args.notes}
    append_csv(log_dir, row, fields)
    print(f"CSV → {csv_path(log_dir)}")
if __name__ == "__main__":
    main()


Overwriting /content/xgcca_ssg/src/main.py


In [30]:
%%writefile /content/xgcca_ssg/plots/sensitivity.py
import os, argparse, numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
def ensure_out(d): os.makedirs(d, exist_ok=True)
def read_csv(p):
    df = pd.read_csv(p)
    for c in ["val_acc","test_acc","auroc","avg_sec_per_epoch","total_train_sec","selected_dims",
              "sparsity_lambda","tau","num_heads","dfr","der","hid_dim","out_dim","n_layers","lambd","lr1","lr2","wd1","wd2"]:
        if c in df.columns: df[c] = pd.to_numeric(df[c], errors="coerce")
    return df
def agg(df, by, metric):
    g = df.groupby(by, dropna=True)
    out = g.agg({metric:"mean","selected_dims":"mean","avg_sec_per_epoch":"mean"}).reset_index()
    return out.sort_values(by)
def plot_xy(x,y,xlabel,ylabel,title,out):
    plt.figure(); plt.plot(x,y,marker="o"); plt.xlabel(xlabel); plt.ylabel(ylabel); plt.title(title); plt.tight_layout()
    plt.savefig(out+".pdf"); plt.savefig(out+".png"); plt.close()
def plot_twin(x,y1,y2,xlabel,l1,l2,title,out):
    fig,ax1=plt.subplots(); ax2=ax1.twinx(); ax1.plot(x,y1,marker="o"); ax2.plot(x,y2,marker="s")
    ax1.set_xlabel(xlabel); ax1.set_ylabel(l1); ax2.set_ylabel(l2); ax1.set_title(title)
    fig.tight_layout(); fig.savefig(out+".pdf"); fig.savefig(out+".png"); plt.close(fig)
def main():
    ap = argparse.ArgumentParser(); ap.add_argument("--csv", required=True); ap.add_argument("--out", default="plots_out")
    ap.add_argument("--metric", default="test_acc", choices=["test_acc","val_acc"]); args = ap.parse_args()
    ensure_out(args.out); df = read_csv(args.csv); m=args.metric
    if "sparsity_lambda" in df.columns:
        a=agg(df.dropna(subset=["sparsity_lambda"]),"sparsity_lambda",m)
        if len(a): plot_twin(a["sparsity_lambda"], a[m], a["selected_dims"], "sparsity_lambda (γ)", m.replace("_"," ").title(), "Selected dims |S|", "Accuracy and |S| vs γ", os.path.join(args.out,"acc_vs_gamma"))
    if "tau" in df.columns:
        a=agg(df.dropna(subset=["tau"]),"tau",m)
        if len(a): plot_xy(a["tau"], a[m], "tau", m.replace("_"," ").title(), "Accuracy vs tau", os.path.join(args.out,"acc_vs_tau"))
    if "num_heads" in df.columns:
        a=agg(df.dropna(subset=["num_heads"]),"num_heads",m)
        if len(a): plot_xy(a["num_heads"], a[m], "num_heads", m.replace("_"," ").title(), "Accuracy vs num_heads", os.path.join(args.out,"acc_vs_heads"))
    if "dfr" in df.columns:
        a=agg(df.dropna(subset=["dfr"]),"dfr",m)
        if len(a): plot_xy(a["dfr"], a[m], "feature dropout (dfr)", m.replace("_"," ").title(), "Accuracy vs feature dropout", os.path.join(args.out,"acc_vs_dfr"))
    if set(["avg_sec_per_epoch","selected_dims"]).issubset(df.columns):
        d = df[["avg_sec_per_epoch","selected_dims"]].dropna()
        if len(d): plot_xy(d["selected_dims"], d["avg_sec_per_epoch"], "Selected dims |S|","Avg sec/epoch","Runtime vs |S|", os.path.join(args.out,"runtime_vs_selected_dims"))
    if "auroc" in df.columns:
        for key,name in [("sparsity_lambda","γ"),("tau","tau"),("num_heads","num_heads"),("dfr","dfr")]:
            if key in df.columns:
                a = agg(df.dropna(subset=[key]), key, "auroc")
                if len(a): plot_xy(a[key], a["auroc"], key, "AUROC", f"AUROC vs {name}", os.path.join(args.out, f"auroc_vs_{key}"))
if __name__ == "__main__": main()


Overwriting /content/xgcca_ssg/plots/sensitivity.py


In [32]:
# Create missing folders and (re)write the ablation scripts
import os, textwrap, pathlib
base = "/content/xgcca_ssg"
pathlib.Path(f"{base}/scripts").mkdir(parents=True, exist_ok=True)
pathlib.Path(f"{base}/plots").mkdir(parents=True, exist_ok=True)

In [33]:
%%writefile /content/xgcca_ssg/scripts/ablation.py
import argparse, os, subprocess, sys, json, time, csv, glob
from datetime import datetime

SUPPORTED = ["cora","citeseer","pubmed","photo","comp","cs","physics","arxiv","reddit"]

def run_one(dataname, base_args):
    # Construct CLI
    args = ["python","-m","src.main","--dataname",dataname]
    # fixed hyperparams (you can extend/override from base_args)
    defaults = dict(
        epochs=150, hid_dim=128, out_dim=256, n_layers=2, num_heads=4,
        lr1=1e-3, lr2=1e-2, wd1=0.0, wd2=1e-4,
        lambd=1e-4, sparsity_lambda=1e-4, tau=1.0,
        der=0.2, dfr=0.2, seed=42, run_name=f"ablation_{dataname}"
    )
    defaults.update(base_args or {})
    for k,v in defaults.items():
        args += [f"--{k}", str(v)]
    print(">>>", " ".join(args))
    # Run and stream output
    p = subprocess.run(args, check=False)
    return p.returncode

def newest_log_dir():
    logs = sorted(glob.glob("/content/xgcca_ssg/logs/*"), key=os.path.getmtime, reverse=True)
    return logs[0] if logs else None

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--datasets", type=str, default="cora,citeseer,pubmed,photo,comp,cs,physics",
                    help="Comma-separated subset of supported datasets")
    ap.add_argument("--args_json", type=str, default="",
                    help="Optional JSON string to override hyperparameters for all datasets")
    args = ap.parse_args()

    ds = [d.strip() for d in args.datasets.split(",") if d.strip()]
    for d in ds:
        if d not in SUPPORTED:
            print(f"[skip] {d} not supported in current loader.")
    ds = [d for d in ds if d in SUPPORTED]
    if not ds:
        print("No valid datasets requested."); sys.exit(0)

    base_args = json.loads(args.args_json) if args.args_json else {}
    all_rows = []

    for d in ds:
        print(f"\n=== Running dataset: {d} ===")
        rc = run_one(d, base_args)
        if rc != 0:
            print(f"[warn] run failed for {d} (rc={rc}), continuing…")
            continue
        # Grab the most recent sweep_results.csv
        logdir = newest_log_dir()
        if not logdir:
            print(f"[warn] no logdir found for {d}")
            continue
        csv_path = os.path.join(logdir, "sweep_results.csv")
        if not os.path.exists(csv_path):
            print(f"[warn] no sweep_results.csv for {d}")
            continue
        # read last row
        with open(csv_path, newline="") as f:
            rows = list(csv.DictReader(f))
            if rows:
                row = rows[-1].copy()
                row["__logdir__"] = logdir
                row["dataset"] = d
                all_rows.append(row)

    # Save aggregate CSV
    out_dir = "/content/xgcca_ssg/ablation"
    os.makedirs(out_dir, exist_ok=True)
    out_csv = os.path.join(out_dir, f"ablation_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv")
    if all_rows:
        # unify columns
        cols = sorted({k for r in all_rows for k in r.keys()})
        with open(out_csv, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=cols); w.writeheader(); w.writerows(all_rows)
        print(f"[done] Wrote ablation CSV → {out_csv}")
    else:
        print("[warn] nothing collected; no CSV produced.")

if __name__ == "__main__":
    main()


Writing /content/xgcca_ssg/scripts/ablation.py


In [34]:
%%writefile /content/xgcca_ssg/plots/ablation_plots.py
import argparse, os, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def bar_plot(df, col, ylabel, out):
    df = df.copy()
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=[col, "dataset"])
    if df.empty:
        print(f"[skip] {col} empty"); return
    order = df.groupby("dataset")[col].mean().sort_values(ascending=False).index
    plt.figure()
    plt.bar(df["dataset"], df[col])
    plt.ylabel(ylabel); plt.xticks(rotation=45, ha="right"); plt.tight_layout()
    plt.savefig(out+".png"); plt.savefig(out+".pdf"); plt.close()

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", required=True)
    ap.add_argument("--outdir", default="/content/xgcca_ssg/plots_out")
    args = ap.parse_args()

    os.makedirs(args.outdir, exist_ok=True)
    df = pd.read_csv(args.csv)

    # convert % strings if present
    for c in ["val_acc","test_acc","auroc"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # Bars
    if "val_acc" in df.columns:
        bar_plot(df, "val_acc", "Val Accuracy (%)", os.path.join(args.outdir, "ablation_val_acc"))
    if "test_acc" in df.columns:
        bar_plot(df, "test_acc", "Test Accuracy (%)", os.path.join(args.outdir, "ablation_test_acc"))
    if "auroc" in df.columns:
        bar_plot(df, "auroc", "AUROC (%)", os.path.join(args.outdir, "ablation_auroc"))

    # Save a tidy CSV sorted by test acc
    if "test_acc" in df.columns:
        df.sort_values("test_acc", ascending=False).to_csv(os.path.join(args.outdir, "ablation_sorted.csv"), index=False)
        print("[done] Wrote sorted table →", os.path.join(args.outdir, "ablation_sorted.csv"))

if __name__ == "__main__":
    main()


Writing /content/xgcca_ssg/plots/ablation_plots.py


In [43]:
# Create folders
import pathlib
base = "/content/xgcca_ssg"
pathlib.Path(f"{base}/scripts").mkdir(parents=True, exist_ok=True)
pathlib.Path(f"{base}/ablation").mkdir(parents=True, exist_ok=True)
pathlib.Path(f"{base}/plots").mkdir(parents=True, exist_ok=True)

In [44]:


%%writefile /content/xgcca_ssg/scripts/ablation_grid.py
import argparse, os, subprocess, sys, json, time, csv, glob, itertools, random
from datetime import datetime

SUPPORTED = ["cora","citeseer","pubmed","photo","comp","cs","physics","arxiv","reddit"]

def newest_log_dir():
    logs = sorted(glob.glob("/content/xgcca_ssg/logs/*"), key=os.path.getmtime, reverse=True)
    return logs[0] if logs else None

def to_list(v):
    if v is None: return []
    if isinstance(v, (list,tuple)): return list(v)
    return [v]

def grid_from_json(grid_json):
    """
    grid_json is a dict where each value can be:
    - single value (e.g., 120)
    - list of values (e.g., [60,120])
    """
    keys = sorted(grid_json.keys())
    lists = [to_list(grid_json[k]) for k in keys]
    if any(len(lst)==0 for lst in lists):
        raise ValueError("All params in grid_json must be singletons or non-empty lists.")
    for combo in itertools.product(*lists):
        yield dict(zip(keys, combo))

def run_one(dataname, params):
    args = ["python","-m","src.main","--dataname", dataname]
    # add all param key/values as --key value
    # ensure run_name encodes a compact tag
    tag = "_".join([f"{k}={params[k]}" for k in sorted(params.keys()) if k not in ["notes","run_name"]])
    run_name = params.get("run_name", f"grid_{dataname}_{tag.replace(' ','')}")
    merged = dict(params, run_name=run_name)
    for k,v in merged.items():
        args += [f"--{k}", str(v)]
    print(">>>", " ".join(args))
    rc = subprocess.run(args, check=False).returncode
    if rc != 0:
        print(f"[warn] run failed rc={rc}")
        return None
    logdir = newest_log_dir()
    if not logdir:
        print("[warn] no logdir found")
        return None
    csv_path = os.path.join(logdir, "sweep_results.csv")
    if not os.path.exists(csv_path):
        print("[warn] no sweep_results.csv at", csv_path)
        return None
    with open(csv_path, newline="") as f:
        rows = list(csv.DictReader(f))
        if not rows:
            return None
        row = rows[-1].copy()
        row["__logdir__"] = logdir
        return row

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--datasets", type=str, default="cora,citeseer,pubmed,photo,comp,cs,physics")
    ap.add_argument("--grid_json", type=str, required=True,
                    help="JSON dict of param lists/singletons for: epochs,sparsity_lambda,tau,hid_dim,out_dim,n_layers,num_heads,der,dfr (and optional lr1/lr2/wd1/wd2/seed/notes)")
    ap.add_argument("--shuffle", action="store_true", help="shuffle the grid")
    ap.add_argument("--max_runs", type=int, default=0, help="cap total runs (0 = no cap)")
    args = ap.parse_args()

    ds = [d.strip() for d in args.datasets.split(",") if d.strip()]
    ds = [d for d in ds if d in SUPPORTED]
    if not ds:
        print("No valid datasets."); sys.exit(0)

    grid = json.loads(args.grid_json)
    combos = list(grid_from_json(grid))
    if args.shuffle:
        random.shuffle(combos)
    if args.max_runs > 0:
        combos = combos[:args.max_runs]

    print(f"[info] datasets={ds}")
    print(f"[info] total combos={len(combos)}")

    collected = []
    for d in ds:
        for p in combos:
            print(f"\n=== dataset={d} | params={p} ===")
            row = run_one(d, p)
            if row is None:
                continue
            row["dataset"] = d
            # Keep all param settings in the row
            for k,v in p.items():
                row[str(k)] = v
            collected.append(row)

    out_dir = "/content/xgcca_ssg/ablation"
    os.makedirs(out_dir, exist_ok=True)
    out_csv = os.path.join(out_dir, f"ablation_grid_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv")
    if collected:
        cols = sorted({k for r in collected for k in r.keys()})
        with open(out_csv, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=cols)
            w.writeheader(); w.writerows(collected)
        print("[done] wrote:", out_csv)
    else:
        print("[warn] nothing collected")

if __name__ == "__main__":
    main()


Writing /content/xgcca_ssg/scripts/ablation_grid.py


## Train a single run (Cora)

In [45]:
%%writefile /content/xgcca_ssg/plots/ablation_grid_plots.py
import argparse, os, pandas as pd, numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def best_per_dataset(df, metric="test_acc"):
    df = df.copy()
    df[metric] = pd.to_numeric(df[metric], errors="coerce")
    df = df.dropna(subset=[metric, "dataset"])
    # take the max metric per dataset
    idx = df.groupby("dataset")[metric].idxmax()
    return df.loc[idx].sort_values(metric, ascending=False)

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", required=True)
    ap.add_argument("--outdir", default="/content/xgcca_ssg/plots_out")
    ap.add_argument("--metric", default="test_acc")
    args = ap.parse_args()

    os.makedirs(args.outdir, exist_ok=True)
    df = pd.read_csv(args.csv)

    # convert % strings if present
    for c in ["val_acc","test_acc","auroc"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    best = best_per_dataset(df, args.metric)
    if best.empty:
        print("[warn] no rows to plot"); return

    # save a tidy CSV of best rows
    out_csv = os.path.join(args.outdir, f"best_by_dataset_{args.metric}.csv")
    best.to_csv(out_csv, index=False)
    print("[done] wrote:", out_csv)

    # bar plot
    plt.figure()
    plt.bar(best["dataset"], pd.to_numeric(best[args.metric], errors="coerce"))
    plt.ylabel(args.metric.replace("_"," ").title()+" (%)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    out_png = os.path.join(args.outdir, f"best_by_dataset_{args.metric}.png")
    out_pdf = os.path.join(args.outdir, f"best_by_dataset_{args.metric}.pdf")
    plt.savefig(out_png); plt.savefig(out_pdf); plt.close()
    print("[done] plots:", out_png, "and .pdf")

if __name__ == "__main__":
    main()


Writing /content/xgcca_ssg/plots/ablation_grid_plots.py


In [24]:
%cd /content/xgcca_ssg
!python -m src.main --dataname citeseer --epochs 150 --run_name citeseer_base


/content/xgcca_ssg
/root/.dgl/citeseer.zip: 100% 239k/239k [00:00<00:00, 12.9MB/s]
Extracting file to /root/.dgl/citeseer_d6836239
Finished data loading and preprocessing.
  NumNodes: 3327
  NumEdges: 9228
  NumFeats: 3703
  NumClasses: 6
  NumTrainingSamples: 120
  NumValidationSamples: 500
  NumTestSamples: 1000
Done saving data into cached files.
Epoch 001 | loss -152.4245 | |S| 161 | 0.329s
Epoch 002 | loss -180.1592 | |S| 158 | 0.019s
Epoch 003 | loss -181.7029 | |S| 159 | 0.018s
Epoch 004 | loss -195.7038 | |S| 167 | 0.018s
Epoch 005 | loss -202.1219 | |S| 174 | 0.017s
Epoch 006 | loss -204.9659 | |S| 173 | 0.017s
Epoch 007 | loss -216.8963 | |S| 162 | 0.017s
Epoch 008 | loss -215.3649 | |S| 157 | 0.017s
Epoch 009 | loss -222.4117 | |S| 155 | 0.017s
Epoch 010 | loss -225.6671 | |S| 157 | 0.017s
Epoch 011 | loss -224.4450 | |S| 175 | 0.017s
Epoch 012 | loss -227.9647 | |S| 154 | 0.017s
Epoch 013 | loss -229.6298 | |S| 160 | 0.016s
Epoch 014 | loss -233.8915 | |S| 169 | 0.016s
Epoc

## Optional: One-vs-rest AUROC demo (Cora: class 0 positive)

In [17]:
!python -m src.main --dataname cora --epochs 120 --binary_one_vs_rest --binary_pos_class 0 --run_name cora_bin_c0


  NumNodes: 2708
  NumEdges: 10556
  NumFeats: 1433
  NumClasses: 7
  NumTrainingSamples: 140
  NumValidationSamples: 500
  NumTestSamples: 1000
Done loading data from cached files.
[OvR] pos_class=0 | train pos rate=0.143
Epoch 001 | loss -171.0829 | |S| 161 | 0.309s
Epoch 002 | loss -178.8509 | |S| 158 | 0.016s
Epoch 003 | loss -184.2815 | |S| 159 | 0.014s
Epoch 004 | loss -191.9296 | |S| 167 | 0.015s
Epoch 005 | loss -187.4283 | |S| 174 | 0.015s
Epoch 006 | loss -183.9905 | |S| 173 | 0.015s
Epoch 007 | loss -208.4628 | |S| 162 | 0.014s
Epoch 008 | loss -202.3409 | |S| 157 | 0.014s
Epoch 009 | loss -214.5148 | |S| 155 | 0.014s
Epoch 010 | loss -204.4213 | |S| 157 | 0.023s
Epoch 011 | loss -214.2899 | |S| 175 | 0.026s
Epoch 012 | loss -217.3015 | |S| 154 | 0.022s
Epoch 013 | loss -218.3307 | |S| 160 | 0.025s
Epoch 014 | loss -219.8435 | |S| 169 | 0.027s
Epoch 015 | loss -225.9685 | |S| 171 | 0.015s
Epoch 016 | loss -224.5295 | |S| 168 | 0.016s
Epoch 017 | loss -228.0166 | |S| 173 | 0.

In [35]:
# Datasets to compare (trim if you want it faster)
DATASETS = "cora,citeseer,pubmed,photo,comp,cs,physics"

# Optional: override defaults (epochs, sparsity, etc.) for ALL datasets at once
OVERRIDES = {
    "epochs": 120,
    "sparsity_lambda": 1e-4,
    "tau": 1.0,
    "hid_dim": 128,
    "out_dim": 256,
    "n_layers": 2,
    "num_heads": 4,
    "der": 0.2,
    "dfr": 0.2,
    "seed": 42
}

import json, glob, os
%cd /content/xgcca_ssg
!python scripts/ablation.py --datasets {DATASETS} --args_json '{json.dumps(OVERRIDES)}'

# Find the newest ablation CSV and plot
ablations = sorted(glob.glob("/content/xgcca_ssg/ablation/*.csv"), key=os.path.getmtime, reverse=True)
ABL_CSV = ablations[0]
print("Using:", ABL_CSV)
!python plots/ablation_plots.py --csv $ABL_CSV --outdir /content/xgcca_ssg/plots_out

print("Plots & tables → /content/xgcca_ssg/plots_out")


/content/xgcca_ssg

=== Running dataset: cora ===
>>> python -m src.main --dataname cora --epochs 120 --hid_dim 128 --out_dim 256 --n_layers 2 --num_heads 4 --lr1 0.001 --lr2 0.01 --wd1 0.0 --wd2 0.0001 --lambd 0.0001 --sparsity_lambda 0.0001 --tau 1.0 --der 0.2 --dfr 0.2 --seed 42 --run_name ablation_cora
  NumNodes: 2708
  NumEdges: 10556
  NumFeats: 1433
  NumClasses: 7
  NumTrainingSamples: 140
  NumValidationSamples: 500
  NumTestSamples: 1000
Done loading data from cached files.
Epoch 001 | loss -171.0829 | |S| 161 | 0.439s
Epoch 002 | loss -178.8509 | |S| 158 | 0.016s
Epoch 003 | loss -184.2814 | |S| 159 | 0.014s
Epoch 004 | loss -191.9296 | |S| 167 | 0.014s
Epoch 005 | loss -187.4283 | |S| 174 | 0.014s
Epoch 006 | loss -183.9905 | |S| 173 | 0.015s
Epoch 007 | loss -208.4628 | |S| 162 | 0.014s
Epoch 008 | loss -202.3409 | |S| 157 | 0.014s
Epoch 009 | loss -214.5148 | |S| 155 | 0.014s
Epoch 010 | loss -204.4213 | |S| 157 | 0.014s
Epoch 011 | loss -214.2899 | |S| 175 | 0.014s
Epoc

In [ ]:
import json, glob, os
%cd /content/xgcca_ssg

DATASETS = "cora,citeseer,pubmed,photo,comp,cs,physics"

GRID = {
    "epochs": [60, 100, 120],              # training length
    "sparsity_lambda": [1e-5, 1e-4, 5e-4],
    "tau": [0.5, 0.8, 1.0],                # gumbel temperature
    "hid_dim": [128],                 # encoder hidden
    "out_dim": [256],                 # projection dim
    "n_layers": [2],                  # GNN depth
    "num_heads": [1, 2, 4, 8],              # attention heads
    "der": [0.1, 0.2],                # edge drop
    "dfr": [0.1, 0.2],                # feature drop

    # (optional—unchanged if omitted)
    "lr1": 1e-3, "lr2": 1e-2, "wd1": 0.0, "wd2": 1e-4,
    "seed": 42, "notes": "grid-ablation"
}

# Shrink total runs if needed (0 = full grid)
MAX_RUNS = 0
SHUFFLE = False

!python scripts/ablation_grid.py \
  --datasets "{DATASETS}" \
  --grid_json '{json.dumps(GRID)}' \
  --max_runs {MAX_RUNS} \
  --shuffle


Streaming output truncated to the last 5000 lines.
Epoch 095 | loss -249.5083 | |S| 175 | 0.016s
Epoch 096 | loss -249.1068 | |S| 174 | 0.017s
Epoch 097 | loss -249.1401 | |S| 167 | 0.019s
Epoch 098 | loss -249.2308 | |S| 166 | 0.016s
Epoch 099 | loss -249.2683 | |S| 174 | 0.016s
Epoch 100 | loss -249.3220 | |S| 171 | 0.016s
Saved checkpoint → /content/xgcca_ssg/logs/citeseer_20251024_224021/model_epoch_100.pth
Epoch 101 | loss -249.0450 | |S| 166 | 0.016s
Epoch 102 | loss -249.3746 | |S| 160 | 0.016s
Epoch 103 | loss -249.5839 | |S| 173 | 0.016s
Epoch 104 | loss -249.3624 | |S| 179 | 0.016s
Epoch 105 | loss -249.7496 | |S| 150 | 0.016s
Epoch 106 | loss -249.3997 | |S| 167 | 0.016s
Epoch 107 | loss -249.6872 | |S| 161 | 0.016s
Epoch 108 | loss -249.4024 | |S| 153 | 0.016s
Epoch 109 | loss -249.5387 | |S| 164 | 0.018s
Epoch 110 | loss -249.6174 | |S| 149 | 0.016s
Epoch 111 | loss -249.3753 | |S| 145 | 0.018s
Epoch 112 | loss -249.5794 | |S| 161 | 0.023s
Epoch 113 | loss -249.8440 | |S| 

In [ ]:
# Find newest aggregate CSV and plot best test_acc per dataset
ablations = sorted(glob.glob("/content/xgcca_ssg/ablation/ablation_grid_*.csv"), key=os.path.getmtime, reverse=True)
ABL_CSV = ablations[0]
print("Using:", ABL_CSV)

!python plots/ablation_grid_plots.py --csv $ABL_CSV --metric test_acc --outdir /content/xgcca_ssg/plots_out

print("Outputs:")
print(" • Best-by-dataset table:", "/content/xgcca_ssg/plots_out/best_by_dataset_test_acc.csv")
print(" • Plot:", "/content/xgcca_ssg/plots_out/best_by_dataset_test_acc.png (and .pdf)")


## Small γ sweep (sparsity strength)

In [18]:
for g in [1e-6, 3e-6, 1e-5, 3e-5, 1e-4]:
    !python -m src.main --dataname cora --epochs 120 --sparsity_lambda {g} --run_name gamma_{g}


  NumNodes: 2708
  NumEdges: 10556
  NumFeats: 1433
  NumClasses: 7
  NumTrainingSamples: 140
  NumValidationSamples: 500
  NumTestSamples: 1000
Done loading data from cached files.
Epoch 001 | loss -171.0829 | |S| 161 | 0.332s
Epoch 002 | loss -178.8510 | |S| 158 | 0.016s
Epoch 003 | loss -184.2815 | |S| 159 | 0.016s
Epoch 004 | loss -191.9296 | |S| 167 | 0.015s
Epoch 005 | loss -187.4283 | |S| 174 | 0.014s
Epoch 006 | loss -183.9905 | |S| 173 | 0.015s
Epoch 007 | loss -208.4629 | |S| 162 | 0.015s
Epoch 008 | loss -202.3409 | |S| 157 | 0.014s
Epoch 009 | loss -214.5148 | |S| 155 | 0.016s
Epoch 010 | loss -204.4214 | |S| 157 | 0.014s
Epoch 011 | loss -214.2900 | |S| 175 | 0.015s
Epoch 012 | loss -217.3016 | |S| 154 | 0.016s
Epoch 013 | loss -218.3308 | |S| 160 | 0.015s
Epoch 014 | loss -219.8436 | |S| 169 | 0.015s
Epoch 015 | loss -225.9685 | |S| 171 | 0.014s
Epoch 016 | loss -224.5295 | |S| 168 | 0.014s
Epoch 017 | loss -228.0167 | |S| 173 | 0.014s
Epoch 018 | loss -225.6986 | |S| 166

## Generate paper figures from latest run

In [19]:
import glob, os
runs = sorted(glob.glob('/content/xgcca_ssg/logs/*'), key=os.path.getmtime, reverse=True)
RUN_DIR = runs[0]
print('Using run:', RUN_DIR)
!python /content/xgcca_ssg/plots/sensitivity.py --csv $RUN_DIR/sweep_results.csv --out /content/xgcca_ssg/plots_out --metric test_acc
print('Figures saved in /content/xgcca_ssg/plots_out')


Using run: /content/xgcca_ssg/logs/cora_20251024_201355
Figures saved in /content/xgcca_ssg/plots_out
